# Построение и оценка моделей

## Цель

На этом этапе необходимо построить модели для прогнозирования вероятности
возникновения серьёзной просрочки по кредитным обязательствам.

Основные задачи:

- сформировать baseline;
- обучить несколько моделей классификации;
- оценить их качество на валидационной выборке;
- сравнить модели по выбранным метрикам;
- выбрать наиболее перспективную модель;
- выполнить настройку её гиперпараметров;
- провести финальную оценку.

В качестве основных метрик используются ROC-AUC и PR-AUC.
Дополнительно рассматриваются precision, recall и F1-score.

In [ ]:
# необходимые импорты 
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
import sklearn
sklearn.set_config(display="text")

# Добавляем корень проекта в Python path
sys.path.append(str(Path.cwd().parent))

from src.preprocessing import preprocessor

In [ ]:
# загружаем данные 

train = pd.read_csv("../data/raw/cs-training.csv")

print(f"Размер датасета: {train.shape}")

Размер датасета: (150000, 12)


In [ ]:
# выделяем целевую переменную 

X = train.drop(columns=["SeriousDlqin2yrs", "Unnamed: 0"])
y = train["SeriousDlqin2yrs"]

In [ ]:
# проверка 

print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("Доля положительного класса:", y.mean())

Размер X: (150000, 10)
Размер y: (150000,)
Доля положительного класса: 0.06684


In [ ]:
# делим на выборки 

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# проверка 

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

print("\nДоля положительного класса:")
print("Train:", y_train.mean())
print("Validation:", y_valid.mean())

Train: (120000, 10)
Validation: (30000, 10)

Доля положительного класса:
Train: 0.06684166666666666
Validation: 0.06683333333333333


## 3. Baseline

В качестве baseline используем `DummyClassifier` со стратегией `prior`.

Модель не использует признаки объектов и предсказывает вероятность положительного
класса на основе его доли в обучающей выборке.

Baseline нужен как контрольная точка для оценки того, насколько реальные модели
извлекают полезную информацию из признаков.

In [ ]:
# обучаем baseline 

baseline = DummyClassifier(
    strategy="prior",
)

baseline.fit(X_train, y_train)

DummyClassifier()

In [ ]:
# получаем вероятности 

baseline_proba = baseline.predict_proba(X_valid)[:, 1]

In [ ]:
# оцениваем baseline

baseline_roc_auc = roc_auc_score(
    y_valid,
    baseline_proba
)

baseline_pr_auc = average_precision_score(
    y_valid,
    baseline_proba
)

print(f"ROC-AUC: {baseline_roc_auc:.4f}")
print(f"PR-AUC: {baseline_pr_auc:.4f}")

ROC-AUC: 0.5000
PR-AUC: 0.0668


## 4. Logistic Regression

Логистическая регрессия используется в качестве первой базовой модели классификации.

Она позволяет получить интерпретируемую линейную зависимость между признаками
и вероятностью возникновения целевого события.

Preprocessing и модель объединяются в единый Pipeline.

In [ ]:
# создаем pipeline

logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])

In [ ]:
# обучаем модель

logreg_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('invalid_values', ReplaceInvalidValues()),
                                 ('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model', LogisticRegression(max_iter=1000))])

In [ ]:
# получаем вероятности 

logreg_proba = logreg_pipeline.predict_proba(X_valid)[:, 1]

In [ ]:
# оцениваем модель 

logreg_roc_auc = roc_auc_score(
    y_valid,
    logreg_proba
)

logreg_pr_auc = average_precision_score(
    y_valid,
    logreg_proba
)

print(f"ROC-AUC: {logreg_roc_auc:.4f}")
print(f"PR-AUC: {logreg_pr_auc:.4f}")

ROC-AUC: 0.8156
PR-AUC: 0.3521


## 5. Random Forest

Random Forest используется для проверки того, позволяет ли нелинейная модель
получить более высокое качество по сравнению с Logistic Regression.

Модель обучается внутри Pipeline вместе с preprocessing.

In [ ]:
# создаем pipeline

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# обучаем модель

rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('invalid_values', ReplaceInvalidValues()),
                                 ('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model',
                 RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                        random_state=42))])

In [ ]:
# получаем вероятности 

rf_proba = rf_pipeline.predict_proba(X_valid)[:, 1]

In [ ]:
# оцениваем модель

rf_roc_auc = roc_auc_score(
    y_valid,
    rf_proba
)

rf_pr_auc = average_precision_score(
    y_valid,
    rf_proba
)

print(f"ROC-AUC: {rf_roc_auc:.4f}")
print(f"PR-AUC: {rf_pr_auc:.4f}")

ROC-AUC: 0.8482
PR-AUC: 0.3628


## 6. CatBoost

CatBoost используется для сравнения с Logistic Regression и Random Forest.

Модель позволяет проверить, даст ли современный градиентный бустинг дополнительное
качество на задаче кредитного скоринга.

Preprocessing и модель объединяются в единый Pipeline.

In [ ]:
#создаем pipeline

catboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False
    ))
])

In [ ]:
# обучаем модель

catboost_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('invalid_values', ReplaceInvalidValues()),
                                 ('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model',
                 CatBoostClassifier(depth=6, eval_metric='AUC', iterations=300, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False))])

In [ ]:
# получаем вероятности 

catboost_proba = catboost_pipeline.predict_proba(X_valid)[:, 1]


In [ ]:
# оцениваем модель

catboost_roc_auc = roc_auc_score(
    y_valid,
    catboost_proba
)

catboost_pr_auc = average_precision_score(
    y_valid,
    catboost_proba
)

print(f"ROC-AUC: {catboost_roc_auc:.4f}")
print(f"PR-AUC: {catboost_pr_auc:.4f}")

ROC-AUC: 0.8696
PR-AUC: 0.4048


## 7. Сравнение моделей

Сравним полученные модели по ROC-AUC и PR-AUC.

ROC-AUC показывает способность модели ранжировать объекты по вероятности
положительного класса.

PR-AUC особенно информативна в данной задаче из-за дисбаланса целевой переменной.

In [ ]:
# собираем все результаты в таблицу

results = pd.DataFrame({
    "model": [
        "DummyClassifier",
        "Logistic Regression",
        "Random Forest",
        "CatBoost"
    ],
    "ROC-AUC": [
        baseline_roc_auc,
        logreg_roc_auc,
        rf_roc_auc,
        catboost_roc_auc
    ],
    "PR-AUC": [
        baseline_pr_auc,
        logreg_pr_auc,
        rf_pr_auc,
        catboost_pr_auc
    ]
})

results.sort_values("PR-AUC", ascending=False)

,model,ROC-AUC,PR-AUC
3,CatBoost,0.869579,0.404809
2,Random Forest,0.848164,0.362766
1,Logistic Regression,0.815552,0.352117
0,DummyClassifier,0.500000,0.066833


## 8. Подбор гиперпараметров CatBoost

Для оптимизации гиперпараметров используется Optuna.

Оптимизация проводится только на обучающей выборке с использованием
кросс-валидации. Валидационная выборка не используется при подборе
гиперпараметров и сохраняется для финальной оценки модели.

In [ ]:
# функция objectiv для optuna

def objective(trial):

    params = {
        "iterations": trial.suggest_int(
            "iterations",
            200,
            500
        ),
        "depth": trial.suggest_int(
            "depth",
            4,
            8
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.15,
            log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            10,
            log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            2
        ),
    }

    cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
    )

    scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_valid = X_train.iloc[valid_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_valid = y_train.iloc[valid_idx]

        model = Pipeline([
            ("preprocessor", preprocessor),
            ("model", CatBoostClassifier(
                **params,
                loss_function="Logloss",
                eval_metric="AUC",
                random_seed=42,
                verbose=False
            ))
        ])

        model.fit(X_fold_train, y_fold_train)

        proba = model.predict_proba(X_fold_valid)[:, 1]

        score = roc_auc_score(
            y_fold_valid,
            proba
        )

        scores.append(score)

    return np.mean(scores)

In [ ]:
# зупскаем optuna 

study = optuna.create_study(
    direction="maximize",
    study_name="catboost_credit_scoring"
)

study.optimize(
    objective,
    n_trials=15
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

[I 2026-08-18 12:27:49,262] A new study created in memory with name: catboost_credit_scoring
[I 2026-08-18 12:28:06,431] Trial 0 finished with value: 0.8598201678487765 and parameters: {'iterations': 412, 'depth': 4, 'learning_rate': 0.012519997401771189, 'l2_leaf_reg': 4.692271166402139, 'random_strength': 1.1902992054757573}. Best is trial 0 with value: 0.8598201678487765.
[I 2026-08-18 12:28:26,022] Trial 1 finished with value: 0.8628573857600892 and parameters: {'iterations': 286, 'depth': 8, 'learning_rate': 0.04447820323094756, 'l2_leaf_reg': 1.9385249362433667, 'random_strength': 1.6221220946338966}. Best is trial 1 with value: 0.8628573857600892.
[I 2026-08-18 12:28:48,015] Trial 2 finished with value: 0.8611228061305971 and parameters: {'iterations': 325, 'depth': 8, 'learning_rate': 0.010565821228633062, 'l2_leaf_reg': 1.2292232093229942, 'random_strength': 1.790129724176801}. Best is trial 1 with value: 0.8628573857600892.
[I 2026-08-18 12:29:03,098] Trial 3 finished with va

In [ ]:
# проверяем результаты 

print("Лучший ROC-AUC:", study.best_value)

print("\nЛучшие параметры:")
for param, value in study.best_params.items():
    print(f"{param}: {value}")

Лучший ROC-AUC: 0.8631213733863656

Лучшие параметры:
iterations: 295
depth: 6
learning_rate: 0.05174578677421144
l2_leaf_reg: 4.24818897663055
random_strength: 1.1939479629869374


## 9. Финальная модель

На основании результатов Optuna выбираются лучшие гиперпараметры CatBoost.

Финальная модель обучается на всей обучающей выборке (`X_train`, `y_train`).

Валидационная выборка (`X_valid`, `y_valid`) не использовалась при подборе
гиперпараметров и применяется только для финальной оценки качества модели.

In [ ]:
# создаем финальный пайплайн

final_catboost = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostClassifier(
        **study.best_params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False
    ))
])

In [ ]:
# обучаем на полном наборе данных 

final_catboost.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('invalid_values', ReplaceInvalidValues()),
                                 ('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model',
                 CatBoostClassifier(depth=6, eval_metric='AUC', iterations=295, l2_leaf_reg=4.24818897663055, learning_rate=0.05174578677421144, loss_function='Logloss', random_seed=42, random_strength=1.1939479629869374, verbose=False))])

In [ ]:
# получаем вероятности 

final_proba = final_catboost.predict_proba(X_valid)[:, 1]

In [ ]:
# оцениваем модель 

final_roc_auc = roc_auc_score(
    y_valid,
    final_proba
)

final_pr_auc = average_precision_score(
    y_valid,
    final_proba
)

print(f"ROC-AUC: {final_roc_auc:.4f}")
print(f"PR-AUC: {final_pr_auc:.4f}")

ROC-AUC: 0.8695
PR-AUC: 0.4047


## 10. Оценка классификации

Помимо ROC-AUC и PR-AUC, оценим работу финальной модели как бинарного
классификатора.

Для этого преобразуем предсказанные вероятности в классы с использованием
порога 0.5 и рассчитаем precision, recall и F1-score.

In [ ]:
# получаем классы

threshold = 0.5

final_pred = (final_proba >= threshold).astype(int)

In [ ]:
# расчитываем метрики

final_precision = precision_score(
    y_valid,
    final_pred,
    zero_division=0
)

final_recall = recall_score(
    y_valid,
    final_pred,
    zero_division=0
)

final_f1 = f1_score(
    y_valid,
    final_pred,
    zero_division=0
)

print(f"Threshold: {threshold}")
print(f"Precision: {final_precision:.4f}")
print(f"Recall:    {final_recall:.4f}")
print(f"F1-score:  {final_f1:.4f}")

Threshold: 0.5
Precision: 0.6006
Recall:    0.1845
F1-score:  0.2823


In [ ]:
# смотрим матрицу ошибок

cm = confusion_matrix(y_valid, final_pred)

print("Матрица ошибок:")
print(cm)

Матрица ошибок:
[[27749   246]
 [ 1635   370]]


## 11. Выводы

В рамках этапа моделирования были протестированы четыре подхода:

- DummyClassifier как baseline;
- Logistic Regression;
- Random Forest;
- CatBoost.

Лучший результат на валидационной выборке показал CatBoost.

### Сравнение моделей

| Модель | ROC-AUC | PR-AUC |
|---|---:|---:|
| DummyClassifier | 0.5000 | 0.0668 |
| Logistic Regression | 0.8156 | 0.3521 |
| Random Forest | 0.8482 | 0.3628 |
| CatBoost | 0.8696 | 0.4048 |
| CatBoost после Optuna | 0.8695 | 0.4047 |

Настройка гиперпараметров с помощью Optuna практически не изменила качество
модели на отложенной валидационной выборке. Поэтому исходная конфигурация
CatBoost является достаточной для дальнейшего использования в проекте.

Итоговая модель показывает хорошее качество ранжирования клиентов по риску:
ROC-AUC = 0.8696, PR-AUC = 0.4048.

При стандартном пороге классификации 0.5:

- Precision = 0.6006;
- Recall = 0.1845;
- F1-score = 0.2823.

Выбор оптимального порога не выполнялся, поскольку для него необходимо
задать бизнес-стоимость ошибок первого и второго рода.

Таким образом, CatBoost выбран в качестве финальной модели проекта.